In [12]:
from pathlib import Path
import subprocess
import uuid

import hashlib

WORKSPACE="workspace/repos"

def get_repo_id(repo_url: str) -> str:
    normalized = repo_url.strip().lower().removesuffix(".git")
    return hashlib.sha256(normalized.encode()).hexdigest()[:16]

class RepoCloneError(Exception):
    """Base exception for clone failures."""


class PrivateRepositoryError(RepoCloneError):
    """Raised when the repository is private or requires authentication."""


class RepositoryNotFoundError(RepoCloneError):
    """Raised when the repository does not exist."""


class InvalidRepositoryError(RepoCloneError):
    """Raised when the URL is not a valid git repository."""


class RepoCloneService:
    def __init__(self, workspace: str = "workspace/repos"):
        self.workspace = Path(workspace)
        self.workspace.mkdir(parents=True, exist_ok=True)

    def clone(self, repo_url: str) -> Path:
        repo_id = uuid.uuid4().hex
        repo_path = self.workspace / repo_id

        repo_id = get_repo_id(repo_url)
        repo_path = self.workspace / repo_id

        if repo_path.exists():
            return repo_path

        try:
            subprocess.run(
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "--single-branch",
                    "--filter=blob:none",
                    repo_url,
                    str(repo_path),
                ],
                check=True,
                capture_output=True,
                text=True,
            )

            return repo_path

        except subprocess.CalledProcessError as e:
            error = (e.stderr or "").lower()

            if repo_path.exists():
                import shutil
                shutil.rmtree(repo_path, ignore_errors=True)

            if (
                "authentication failed" in error
                or "could not read username" in error
                or "repository not found" in error
            ):
               
                raise PrivateRepositoryError(
                    "Repository is private or requires authentication."
                )

            if "not appear to be a git repository" in error:
                raise InvalidRepositoryError(
                    "The provided URL is not a valid Git repository."
                )

            if "could not resolve host" in error:
                raise RepoCloneError(
                    "Unable to reach the Git server."
                )

            raise RepoCloneError(
                f"Failed to clone repository.\n{e.stderr.strip()}"
            )

In [ ]:
service = RepoCloneService()



path = service.clone(
    "https://github.com/langchain/langchain"
)

print(path)

workspace\repos\4e38fa06a2890980
